# B.8 - Verifica del Modulo 4 (basi delle reti neurali)

Tempo: 45 minuti. Lavoro individuale. Eseguire le prime due celle, poi completare gli esercizi sostituendo `None`. L'esercizio 6 è una domanda a risposta aperta, da scrivere nella cella di testo indicata.

In [ ]:
# Funzione di controllo degli esercizi: eseguire questa cella, non modificarla.
def controlla(numero, prova):
    try:
        esito = bool(prova())
    except Exception as e:
        print(f"Esercizio {numero}: da rivedere ({type(e).__name__})")
        return
    print(f"Esercizio {numero}: " + ("corretto" if esito else "da rivedere"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def neurone(X, w, b):
    return np.where(X @ w + b >= 0, 1, 0)

**Esercizio 1.** Trovare pesi `w1` (array di 2 valori) e bias `b1` di un neurone a gradino che calcola "x1 OR NOT x2", cioè le uscite `[1, 0, 1, 1]` per i quattro punti di `P`.

In [ ]:
w1 = np.array([1, -1])
b1 = 0.5

In [ ]:
controlla(1, lambda: np.array_equal(neurone(P, w1, b1), [1, 0, 1, 1]))

**Esercizio 2.** Un perceptron ha pesi `w = [0.5, -0.2]`, bias `b = 0.1` e tasso di apprendimento `eta = 0.5`. Riceve l'esempio `x = [1, 1]` con etichetta `y = 0`. Calcolare con il codice la previsione `y_prev`, l'errore `e` e i nuovi valori `w_nuovo` e `b_nuovo` dopo un'applicazione della regola del perceptron.

In [ ]:
w = np.array([0.5, -0.2])
b = 0.1
eta = 0.5
x = np.array([1, 1])
y = 0
y_prev = 1 if x @ w + b >= 0 else 0
e = y - y_prev
w_nuovo = w + eta * e * x
b_nuovo = b + eta * e
print(y_prev, e, w_nuovo, b_nuovo)

In [ ]:
controlla(2, lambda: y_prev == 1 and e == -1 and np.allclose(w_nuovo, [0.0, -0.7]) and np.isclose(b_nuovo, -0.4))

**Esercizio 3.** Applicare 3 passi di discesa del gradiente alla funzione $f(w) = (w + 2)^2$, la cui derivata è $f'(w) = 2(w + 2)$, partendo da $w = 1$ con $\eta = 0.25$. Salvare nella lista `valori_w` il valore iniziale e i tre valori successivi (4 numeri in tutto).

In [ ]:
w_g = 1.0
valori_w = [w_g]
for _ in range(3):
    w_g = w_g - 0.25 * 2 * (w_g + 2)
    valori_w.append(w_g)
print(valori_w)

In [ ]:
controlla(3, lambda: np.allclose(valori_w, [1.0, -0.5, -1.25, -1.625]))

**Esercizio 4.** Calcolare l'uscita `y_rete` della rete con uno strato nascosto per l'ingresso `x_r`: strato nascosto $h = \sigma(W_1 x + b_1)$, uscita $\hat{y} = \sigma(W_2 h + b_2)$.

In [ ]:
W1 = np.array([[2.0, -1.0], [0.5, 1.5], [-1.0, 1.0]])
bb1 = np.array([0.0, -1.0, 0.5])
W2 = np.array([[1.0, -2.0, 0.5]])
bb2 = np.array([0.2])
x_r = np.array([1.0, 2.0])
h = sigmoide(W1 @ x_r + bb1)
y_rete = sigmoide(W2 @ h + bb2)
print(y_rete)

In [ ]:
controlla(4, lambda: np.allclose(y_rete, sigmoide(W2 @ sigmoide(W1 @ x_r + bb1) + bb2)))

**Esercizio 5.** Dati: due cerchi concentrici (classe 0 all'interno, classe 1 all'esterno). Dividere i dati in addestramento (70%) e verifica (30%) con `random_state=0`, addestrare un `MLPClassifier` con uno strato nascosto (scegliere il numero di neuroni e gli altri parametri) e ottenere un'accuratezza sui dati di verifica `acc_verifica` di almeno 0.9.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(5)
angoli = rng.uniform(0, 2 * np.pi, 300)
raggi = np.concatenate([rng.uniform(0, 1, 150), rng.uniform(1.5, 2.5, 150)])
Xc = np.column_stack([raggi * np.cos(angoli), raggi * np.sin(angoli)])
yc = np.concatenate([np.zeros(150), np.ones(150)]).astype(int)

Xa, Xv, ya, yv = train_test_split(Xc, yc, test_size=0.3, random_state=0)
m = MLPClassifier(hidden_layer_sizes=(8,), solver="lbfgs", max_iter=2000, random_state=0)
m.fit(Xa, ya)
acc_verifica = m.score(Xv, yv)
print(acc_verifica)

In [ ]:
controlla(5, lambda: acc_verifica >= 0.9)

**Esercizio 6 (risposta aperta).** In non più di 6 righe: perché un singolo neurone non può separare le due classi dell'esercizio 5, mentre una rete con uno strato nascosto ci riesce? Collegare la risposta all'interpretazione geometrica del neurone.

*Risposta:*